# `c09_sfa` — Student Financial Aid and Net Price

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `SFA2223`, `SFAV2223` |
| Reference period | 2022-23 aid year |
| Curated grain | `UNITID` |
| Output | `data/curated/c09_sfa.parquet` |

The N/P/A triple is a built-in consistency check: the count of recipients, the percentage of the relevant cohort, and the average award. Recomputing the percentage from the count and the denominator should reproduce the published figure, and a mismatch means the wrong denominator was used.

> **Pitfall.** Denominators differ within the same file. Some measures apply to full-time first-time degree-seeking undergraduates (SCUGFFN) and others to all undergraduates (SCUGRAD). Net price series are further restricted to students receiving Title IV aid, so a net-price comparison across institutions with very different aid participation is not comparing the same population.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c09_sfa"
TABLES = ['SFA2223', 'SFAV2223']
GRAIN = ['UNITID']
REFERENCE_PERIOD = '2022-23 aid year'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,SFA2223,1913114,144f97dc24492febc772e2aed6c7eb543037ed56274a49...,2026-09-24T17:19:30+00:00
1,SFAV2223,108372,b125c463e597ea7c998ef8221765c4f094bad2cf7ba2db...,2026-09-24T17:19:30+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'2022-23',
    table=TABLES[0],
)
print(intro[:600])

File Documentation for the Student Financial Aid Data File, 2022-23
(Provisional release)
Filename SFA2223
Overview This file contains data on the number of full-time, first-time degree/certificate-seeking undergraduate students and all undergraduate students who were awarded different types of student financial aid, including grants and loans, from different sources at each institution. Sources and types of aid reported for full-time, first-time degree/certificate-seeking undergraduate students include Federal Pell grants, other federal grants, state/local grants, grants from the institution,


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

346 variables documented, 0 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,SCUGRAD,Total number of undergraduates - financial aid...
2,SCUGDGSK,Total number of degree/certificate-seeking und...
3,SCUGNDGS,Total number of non-degee/non-certificate seek...
4,SCUGFFN,Total number of full-time first-time degree/ce...
5,SCUGFFP,Full-time first-time degree/certificate seekin...
6,SCFA2,Total number of undergraduates - fall cohort
7,SCFA2DG,Total number of degree/certificate-seeking und...
8,SCFA2ND,Total number of non-degee/non-certificate seek...
9,SCFA1N,Number of students in fall cohort


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'SCUGRAD', 'SCUGFFN', 'ANYAIDN', 'ANYAIDP', 'FGRNT_N', 'FGRNT_P', 'FGRNT_A', 'PGRNT_N', 'PGRNT_P', 'PGRNT_A', 'FLOAN_N', 'FLOAN_P', 'FLOAN_A', 'NPIST2', 'NPT412', 'GIS4N12']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (5653, 691)
schema: unchanged | added [] | removed []


,UNITID,SCUGRAD,SCUGFFN,ANYAIDN,ANYAIDP,FGRNT_N,FGRNT_P,FGRNT_A,PGRNT_N,PGRNT_P,PGRNT_A,FLOAN_N,FLOAN_P,FLOAN_A,NPIST2,NPT412,GIS4N12
0,100654,5206,1547,1359.0,88.0,1033.0,67.0,6163.0,1033.0,67.0,5988.0,880.0,57.0,6048.0,14064.0,NaN,328.0
1,100663,13032,2172,2107.0,97.0,891.0,41.0,6264.0,891.0,41.0,5988.0,1087.0,50.0,5121.0,17413.0,NaN,410.0
2,100690,228,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100706,7169,1154,1121.0,97.0,279.0,24.0,5342.0,279.0,24.0,4885.0,432.0,37.0,5090.0,19847.0,NaN,93.0
4,100724,3296,938,862.0,92.0,685.0,73.0,5900.0,685.0,73.0,5521.0,635.0,68.0,5775.0,13504.0,NaN,174.0


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in []:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 0 reserved-code cells across 16 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

,column,flag,n,share
4,XANYAIDN,R,5367,0.9494
5,XANYAIDN,A,282,0.0499
6,XANYAIDN,P,4,0.0007
7,XANYAIDP,R,5367,0.9494
8,XANYAIDP,A,282,0.0499
9,XANYAIDP,P,4,0.0007
20,XFGRNT_A,R,5247,0.9282
21,XFGRNT_A,A,397,0.0702
22,XFGRNT_A,C,5,0.0009
23,XFGRNT_A,P,4,0.0007


Columns under 90% reported — interpret with care:


column
XFLOAN_A    0.8300
XGIS4N12    0.3191
XNPIST2     0.3232
XNPT412     0.5827
Name: share, dtype: float64

## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = []

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,UNITID,SCUGRAD,SCUGFFN,ANYAIDN,ANYAIDP,FGRNT_N,FGRNT_P,FGRNT_A,PGRNT_N,PGRNT_P,PGRNT_A,FLOAN_N,FLOAN_P,FLOAN_A,NPIST2,NPT412,GIS4N12
0,100654,5206.0,1547.0,1359.0,88.0,1033.0,67.0,6163.0,1033.0,67.0,5988.0,880.0,57.0,6048.0,14064.0,NaN,328.0
1,100663,13032.0,2172.0,2107.0,97.0,891.0,41.0,6264.0,891.0,41.0,5988.0,1087.0,50.0,5121.0,17413.0,NaN,410.0
2,100690,228.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100706,7169.0,1154.0,1121.0,97.0,279.0,24.0,5342.0,279.0,24.0,4885.0,432.0,37.0,5090.0,19847.0,NaN,93.0
4,100724,3296.0,938.0,862.0,92.0,685.0,73.0,5900.0,685.0,73.0,5521.0,635.0,68.0,5775.0,13504.0,NaN,174.0


## 9. Reshape to the declared grain

Target grain: `UNITID`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID'] -> 5,653 rows, 0 duplicated


,UNITID,SCUGRAD,SCUGFFN,ANYAIDN,ANYAIDP,FGRNT_N,FGRNT_P,FGRNT_A,PGRNT_N,PGRNT_P,PGRNT_A,FLOAN_N,FLOAN_P,FLOAN_A,NPIST2,NPT412,GIS4N12
0,100654,5206.0,1547.0,1359.0,88.0,1033.0,67.0,6163.0,1033.0,67.0,5988.0,880.0,57.0,6048.0,14064.0,NaN,328.0
1,100663,13032.0,2172.0,2107.0,97.0,891.0,41.0,6264.0,891.0,41.0,5988.0,1087.0,50.0,5121.0,17413.0,NaN,410.0
2,100690,228.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100706,7169.0,1154.0,1121.0,97.0,279.0,24.0,5342.0,279.0,24.0,4885.0,432.0,37.0,5090.0,19847.0,NaN,93.0
4,100724,3296.0,938.0,862.0,92.0,685.0,73.0,5900.0,685.0,73.0,5521.0,635.0,68.0,5775.0,13504.0,NaN,174.0


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID'),
    iu.in_range('ANYAIDP', 0, 100, severity='warn'),
    iu.Rule('pell_le_federal_grant', lambda d: pd.to_numeric(d.PGRNT_N, errors='coerce') > pd.to_numeric(d.FGRNT_N, errors='coerce'), note='Pell recipients are a subset of federal grant recipients'),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,unique_key(UNITID),pass,0,0.0,Declared grain must be unique
1,"in_range(ANYAIDP,0,100)",pass,0,0.0,Value plausibility bound
2,pell_le_federal_grant,pass,0,0.0,Pell recipients are a subset of federal grant ...


PASSED


Report(table='c09_sfa', rows=5653, results=[{'name': 'unique_key(UNITID)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'in_range(ANYAIDP,0,100)', 'severity': 'warn', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}, {'name': 'pell_le_federal_grant', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Pell recipients are a subset of federal grant recipients', 'status': 'pass'}], generated_utc='2026-09-24T17:19:31+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes='Denominators differ within the same file. Some measures apply to full-time first-time degree-seeking undergraduates (SCUGFFN) and others to all undergraduates (SCUGRAD). Net price series are further restricted to students receiving Title IV aid, so a net-price comparison across institutions with very different aid participation is not comparing the same population.',
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c09_sfa.parquet (5,653 rows x 17 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. Denominators differ within the same file. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.